In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load dataset dari file Excel
rule_file = "rule.xlsx"
pattern_file = "dataPenyakit.xlsx"
rule_data = pd.read_excel(rule_file)
pattern_data = pd.read_excel(pattern_file)

In [3]:
# Mengambil daftar penyakit dan gejala dari dataset
disease_list = rule_data['Penyakit'].unique()
all_symptoms = list(set(', '.join(rule_data['Gejala'].dropna()).split(", ")))

In [61]:
# Memilih metode diagnosis
method = input("Pilih metode diagnosis (1: Forward Chaining, 2: Backward Chaining): ")
found = False

# METODE FORWARD CHAINING

In [62]:
if method == '1':
    print("Gejala yang tersedia:")
    for i, symptom in enumerate(all_symptoms):
        print(f"{i+1}. {symptom}")

    # Input gejala dari pengguna
    selected_indexes = input("\nPilih nomor gejala yang Anda alami (pisahkan dengan koma): ")
    selected_indexes = list(map(int, selected_indexes.split(",")))
    selected_symptoms = [all_symptoms[i-1] for i in selected_indexes]

    for index, row in rule_data.iterrows():
        rule_symptoms = row['Gejala'].split(", ")
        if all(symptom in selected_symptoms for symptom in rule_symptoms):
            diagnosis = row['Penyakit']
            medicine = row['Obat']
            # Hasil diagnosa
            print(f"\n[Forward Chaining] Diagnosa: {diagnosis}")
            print(f"Rekomendasi Beberapa Obat: {medicine}")
            found = True
            break

# METODE BACKWARD CHAINING

In [63]:
if method == '2':
    print("\nDaftar penyakit yang tersedia:")
    for i, disease in enumerate(disease_list):
        print(f"{i+1}. {disease}")
    
    # Meminta pengguna memilih penyakit
    selected_index = int(input("\nPilih nomor penyakit yang ingin Anda periksa: "))
    selected_disease = disease_list[selected_index - 1]

    # Menampilkan gejala hanya untuk penyakit yang dipilih
    for index, row in rule_data.iterrows():
        if row['Penyakit'] == selected_disease:
            rule_symptoms = row['Gejala'].split(", ")
            print("Gejala terkait penyakit ini:")
            for i, symptom in enumerate(rule_symptoms):
                print(f"{i+1}. {symptom}")
            
            # Meminta input gejala dari pengguna
            selected_indexes = input("\nPilih nomor gejala yang Anda alami (pisahkan dengan koma): ")
            selected_indexes = list(map(int, selected_indexes.split(",")))
            selected_symptoms = [rule_symptoms[i-1] for i in selected_indexes]

            # Verifikasi gejala
            if all(symptom in selected_symptoms for symptom in rule_symptoms):
                print(f"\n[Backward Chaining] Diagnosa: Penyakit terkonfirmasi [{selected_disease}] berdasarkan gejala.")
                print(f"Rekomendasi Obat: {row['Obat']}")
            else:
                print("\nGejala Anda tidak cocok dengan penyakit ini.")
            break



Daftar penyakit yang tersedia:
1. apnea tidur
2. asbes
3. asma
4. bronkiektasis
5. bronkiolitis
6. bronkitis
7. bronkitis kronis
8. hipertensi paru
9. influensa
10. penyakit aspergilosis
11. penyakit mesothelioma
12. penyakit paru obstruktif kronis
13. pneumotoraks
14. radang paru-paru
15. sindrom kesulitan pernapasan akut
16. tuberkulosis
17. virus sinsitium saluran pernapasan


Gejala terkait penyakit ini:
1. mengi
2. sesak napas
3. kelelahan
4. penurunan berat badan

Gejala Anda tidak cocok dengan penyakit ini.


## Alternatif Pattern Matching dengan cosine similarity

In [64]:
# --- Pattern Matching (Cosine Similarity) ---
if not found:
    print("\nTidak ditemukan diagnosa menggunakan metode yang dipilih.")
    print("Melakukan diagnosa berbasis pattern matching...")
    
    # Menggabungkan gejala pengguna menjadi string
    user_symptoms_str = ", ".join(selected_symptoms)
    
    # Menghitung cosine similarity antara gejala pengguna dan dataset
    vectorizer = CountVectorizer().fit_transform(pattern_data['Gejala'] + [user_symptoms_str])
    vectors = vectorizer.toarray()
    cosine_sim = cosine_similarity(vectors)
    similarities = cosine_sim[-1][:-1]  # Similaritas dengan semua gejala di dataset
    max_index = np.argmax(similarities)
    
    if similarities[max_index] > 0.5:  # Ambang batas similarity
        diagnosis = pattern_data.iloc[max_index]['Penyakit']
        medicine = pattern_data.iloc[max_index]['Obat']
        print(f"\n[Pattern Matching] Berdasarkan gejala yang Anda masukkan, Anda kemungkinan menderita: {diagnosis}")
        print(f"Rekomendasi Obat: {medicine}")
        print("Harap lakukan konsultasi lebih lanjut ke bidang kesehatan.")
    else:
        print("\nGejala Anda terlalu umum atau tidak dapat didiagnosis.")


Tidak ditemukan diagnosa menggunakan metode yang dipilih.
Melakukan diagnosa berbasis pattern matching...

[Pattern Matching] Berdasarkan gejala yang Anda masukkan, Anda kemungkinan menderita: asbes
Rekomendasi Obat: oksigen
Harap lakukan konsultasi lebih lanjut ke bidang kesehatan.
